In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf, pacf

# SPC report 

1. 資料類別： 
- 判斷data類型 -> 在pipeline時會增加判定是 ordinal, nominal, continuous
- Continuous 需要往下做檢驗
- Data 若subgroup > 1 需要先做group by batch

2. 常態

In [ ]:
data = pd.Series(np.random.normal(loc=0, scale=1, size=100))
golen_batches = [0, 1, 2, 3, 4]

In [ ]:
def check_nromality(data: pd.Series, alpha=0.05) -> dict:
    """
    Check if the data follows a normal distribution using the Shapiro-Wilk test.

    Parameters:
    data (array-like): The input data to be tested for normality.
    alpha (float): The significance level for the test (default is 0.05).

    Returns:
    dict: A dictionary containing the Shapiro-Wilk test statistic, p-value, and a boolean indicating whether the null hypothesis (data is normally distributed) is rejected.
    """
    stat, p_value = stats.shapiro(data)
    (osm, osr), _ = stats.probplot(data, dist="norm", plot=None)

    return {"shapiro_stat": stat,
            "shapiro_p_value": p_value,
            "reject_null": p_value < alpha,
            }

In [ ]:
normality_result = check_nromality(data)
print(f"Shapiro-Wilk Test Statistic: {normality_result['shapiro_stat']}")
print(f"Shapiro-Wilk Test P-Value: {normality_result['shapiro_p_value']}")
print(f"Reject Null Hypothesis (Data is not normally distributed): {normality_result['reject_null']}")


In [ ]:
# 2. 建立 1 列 2 欄的子圖畫布
fig, axes = plt.subplots(ncols=2, figsize=(12, 6))

# --- 左圖：Histogram with KDE ---
sns.histplot(data, kde=True, stat="density", ax=axes[0])
axes[0].set_title("Histogram with KDE")
axes[0].set_xlabel("Data Values")
axes[0].set_ylabel("Density")

# --- 右圖：Q-Q Plot ---
# stats.probplot 的 plot 參數可以直接帶入指定的 ax，讓它畫在右邊的子圖上
stats.probplot(data, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot")
axes[1].set_xlabel("Theoretical Quantiles")
axes[1].set_ylabel("Ordered Values")

# 3. 自動調整子圖間距並顯示
plt.tight_layout()
plt.show()

**若非Normal 要先做Transformation 或是 用無母數監控圖**

3. Autocorrelation: ljungbox做n個lags檢定, ACF和PACF做圖

In [ ]:
def autocorrelation_check(data: pd.Series, alpha=0.05) -> dict:
    """
    Check for autocorrelation in the data using the Durbin-Watson test.

    Parameters:
    data (array-like): The input data to be tested for autocorrelation.
    alpha (float): The significance level for the test (default is 0.05).

    Returns:
    dict: A dictionary containing the ljungbox p-value, ACF values, and PACF values.
    """

    lags = min(10, len(data) - 1)

    lb_test = acorr_ljungbox(data, lags=lags, return_df=True)
    lb_p_value = lb_test['lb_pvalue'].values[0]

    acf_values = acf(data, nlags=lags)
    pacf_values = pacf(data, nlags=lags)

    return {
        "ljungbox_p_value": lb_p_value,
        "acf_values": acf_values,
        "pacf_values": pacf_values
    }

In [ ]:
autocorr_result = autocorrelation_check(data)
print(f"Ljung-Box Test P-Value: {autocorr_result['ljungbox_p_value']}")
print(f"ACF Values: {autocorr_result['acf_values']}")
print(f"PACF Values: {autocorr_result['pacf_values']}")

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 6))

sns.barplot(x=np.arange(len(autocorr_result['acf_values'])), y=autocorr_result['acf_values'], ax=axes[0])
axes[0].set_title("Autocorrelation Function (ACF)")
axes[0].set_xlabel("Lag")
axes[0].set_ylabel("ACF Value")

sns.barplot(x=np.arange(len(autocorr_result['pacf_values'])), y=autocorr_result['pacf_values'], ax=axes[1])
axes[1].set_title("Partial Autocorrelation Function (PACF)")
axes[1].set_xlabel("Lag")
axes[1].set_ylabel("PACF Value")

plt.tight_layout()
plt.show()

## Plot

**Batches 數量少於20個建議優先使用spectral limit做完control limit**

In [ ]:
def imr_plot(data: pd.Series) -> dict:
    """
    Generate I-MR (Individuals and Moving Range) control chart parameters.
    param data (array-like): The input data for which the I-MR chart parameters are calculated.
    return dict: A dictionary containing the mean, UCL, and LCL for both the I chart and the MR chart.
    """
    
    # i Chart
    i_mean = np.mean(data)

    # moving range
    mr = np.abs(np.diff(data))
    mr_mean = np.mean(mr)

    # control limits (常數係數：d2 對應之 3/d2 = 2.66, D4 = 3.27)
    i_ucl = i_mean + 2.66 * mr_mean
    i_lcl = max(0, i_mean - 2.66 * mr_mean)  # LCL 不得小於 0

    mr_ucl = 3.27 * mr_mean
    mr_lcl = 0  # MR chart 的 LCL 永遠是 0

    return {
        "i_chart": {
            "mean": i_mean,
            "ucl": i_ucl,
            "lcl": i_lcl
        },
        "mr_chart": {
            "mean": mr_mean,
            "ucl": mr_ucl,
            "lcl": mr_lcl
        }
    }

In [ ]:
def xr_plot(data: pd.Series, subgroup_size: int) -> dict:
    """
    Generate X-bar and R control chart parameters.
    Param data (array-like): The input data for which the X-bar and R chart parameters are calculated.
    Param subgroup_size (int): The size of each subgroup (n).
    Return dict: A dictionary containing the mean, UCL, and LCL for both the X-bar chart and the R chart.
    """

    # 格式: n: (A2, D3, D4)
    CONSTANTS = {
        2: (1.880, 0, 3.267),
        3: (1.023, 0, 2.574),
        4: (0.729, 0, 2.282),
        5: (0.577, 0, 2.114),
        6: (0.483, 0, 2.004),
        7: (0.419, 0.076, 1.924),  # n >= 7 時 D3 不為 0
        8: (0.373, 0.136, 1.864),
        9: (0.337, 0.184, 1.816)
    }
    
    if subgroup_size not in CONSTANTS:
        raise ValueError(f"Subgroup size {subgroup_size} not supported. Supported sizes: {list(CONSTANTS.keys())}")

    A2, D3, D4 = CONSTANTS[subgroup_size]

    # X bar and range
    x_bar = np.mean(data)
    ranges = np.ptp(data)

    # central line
    x_barbar = np.mean(x_bar)
    r_bar = np.mean(ranges)

    # control limits
    x_bar_ucl = x_barbar + A2 * r_bar
    x_bar_lcl = x_barbar - A2 * r_bar

    r_ucl = D4 * r_bar
    r_lcl = D3 * r_bar

    return {
        "x_bar_chart": {
            "mean": x_barbar,
            "ucl": x_bar_ucl,
            "lcl": x_bar_lcl
        },
        "r_chart": {
            "mean": r_bar,
            "ucl": r_ucl,
            "lcl": r_lcl
        }
    }

In [ ]:
def xs_plot(data: pd.Series, subgroup_size: int) -> dict:
    """
    Generate X-s control chart parameters.
    Param data (array-like): The input data for which the X-s chart parameters are calculated.
    Param subgroup_size (int): The size of each subgroup (n).
    Return dict: A dictionary containing the mean, UCL, and LCL for the X-s chart.
    """

    c4 = math.sqrt(2 / (subgroup_size - 1)) * math.gamma(subgroup_size / 2) / math.gamma((subgroup_size - 1) / 2)
    a3 = 3 / c4 * math.sqrt(1 - c4**2)
    b3 = 1 - 3 * c4 / (2 * (subgroup_size - 1))
    b4 = 1 + 3 * c4 / (2 * (subgroup_size - 1))

    # X bar and standard deviation
    x_bar = np.mean(data)
    s = np.std(data, ddof=1)

    # central line
    x_barbar = np.mean(x_bar)
    s_bar = np.mean(s)

    # control limits
    x_bar_ucl = x_barbar + a3 * s_bar
    x_bar_lcl = x_barbar - a3 * s_bar

    s_ucl = b4 * s_bar
    s_lcl = b3 * s_bar

    return {
        "x_bar_chart": {
            "mean": x_barbar,
            "ucl": x_bar_ucl,
            "lcl": x_bar_lcl
        },
        "s_chart": {
            "mean": s_bar,
            "ucl": s_ucl,
            "lcl": s_lcl
        }
    }


In [ ]:
def ewma_plot(data: pd.Series, lambda_: float = 0.2, l = 3) -> dict:
    """
    Generate EWMA (Exponentially Weighted Moving Average) control chart parameters.
    Param data (array-like): The input data for which the EWMA chart parameters are calculated.
    Param lambda_ (float): The smoothing parameter for the EWMA chart (default is 0.2).
    Return dict: A dictionary containing the EWMA values, UCL, and LCL for the EWMA chart.
    """

    # Calculate EWMA values
    ewma_values = [data[0]]  # Initialize with the first data point
    for i in range(1, len(data)):
        ewma_values.append(lambda_ * data[i] + (1 - lambda_) * ewma_values[i - 1])

    ewma_values = np.array(ewma_values)

    # Calculate control limits
    sigma = np.std(data, ddof=1)
    ucl = np.mean(data) + 3 * sigma * np.sqrt(lambda_ / (2 - lambda_))
    lcl = np.mean(data) - 3 * sigma * np.sqrt(lambda_ / (2 - lambda_))

    return {
        "ewma_values": ewma_values,
        "ucl": ucl,
        "lcl": lcl
    }

In [ ]:
def golden_batch_info(data: pd.Series,
                      usl: float, lsl: float,
                      subgroup_size: int,
                      golden_batch_index: list,
                      lambda_: float = 0.2, l: int = 3) -> dict:
    """
    Calculate the golden batch information for the given data.

    Parameters:
    data (array-like): The input data to be analyzed.
    usl (float): The upper specification limit.
    lsl (float): The lower specification limit.
    subgroup_size (int): The size of each subgroup.
    golden_batch_index (list): The indices of the golden batch.

    Returns:
    dict: A dictionary containing the data, upper specification limit, and lower specification limit.
    """
    golden_batch_data = data.iloc[golden_batch_index]

    if subgroup_size <= 0:
        raise ValueError("Subgroup size must be a positive integer.")
    elif subgroup_size == 1:
        shewhart_results = imr_plot(golden_batch_data)
    elif subgroup_size > 1 and subgroup_size < 10:
        shewhart_results = xr_plot(golden_batch_data, subgroup_size)
    else:
        shewhart_results = xs_plot(golden_batch_data, subgroup_size)

    ewma_results = ewma_plot(data, lambda_=lambda_, l=l)

    spectral_limit_results = {"mean": np.mean(golden_batch_data), "usl": usl, "lsl": lsl}
    
    return spectral_limit_results,shewhart_results, ewma_results, golden_batch_index

    

In [ ]:
def cusum_plot(data: pd.Series, target: float = None, k: float = 0.5, h: float = 5.0) -> dict:
    """
    Plot CUSUM chart for monitoring process stability.
    param data (array-like): The input data for which the CUSUM chart is generated.
    param target (float): The target value for the process. If None, the mean of the data is used.
    param k (float): The reference value for the CUSUM chart.
    param h (float): The decision interval for the CUSUM chart.
    return dict: A dictionary containing the CUSUM values (c_plus and c_minus)
    """
    
    mu0 = np.mean(data) if target is None else target

    mr = np.abs(np.diff(data))
    sigma = np.std(mr, ddof=1) / np.sqrt(2)
    if sigma == 0:
        sigma = 1e-10  # Prevent division by zero

    allowance = k * sigma
    decision_limit = h * sigma

    c_plus = np.zeros(len(data))
    c_minus = np.zeros(len(data))

    for i in range(1, len(data)):
        c_plus[i] = max(0, c_plus[i - 1] + (data[i] - mu0) - allowance)
        c_minus[i] = min(0, c_minus[i - 1] + (data[i] - mu0) + allowance)
    
    return {
        "c_plus": c_plus,
        "c_minus": c_minus,
        "decision_limit": decision_limit
    }

In [ ]:
spc_result = golden_batch_info(data, usl=3, lsl=-3, subgroup_size=1, golden_batch_index=[0, 1, 2, 3, 4])
spc_result

In [ ]:
cusum_results = cusum_plot(data, target=None, k=0.5, h=5.0)
cusum_results 

In [ ]:
chart_info.get('mean', spc_result[0]['mean'])

In [ ]:
# 1. 準備顏色陣列（處理異常點變色）
n = len(data)
color = ['blue'] * n
out_indices = [0, 1, 2, 3, 4]  # 你的異常點 index
for i in out_indices:
    color[i] = 'orange'

dynamic_charts = []

# 2. 將 spc_result[0] 的 USL / LSL 全域規格圖加入動態清單
if len(spc_result) > 0 and isinstance(spc_result[0], dict):
    dynamic_charts.append({
        "title": "SPECIFICATION LIMIT CHART (USL/LSL)",
        "x": data.index,
        "y": data.values,
        "mean": spc_result[0]['mean'],
        "upper": spc_result[0].get('usl', spc_result[0].get('ucl')),
        "lower": spc_result[0].get('lsl', spc_result[0].get('lcl'))
    })

# 3. 自動解析傳統管制圖（如 i_chart, mr_chart 等）
if len(spc_result) > 1 and isinstance(spc_result[1], dict):
    for chart_key, chart_info in spc_result[1].items():
        chart_title = chart_key.upper().replace('_', ' ')
        if 'i_chart' in chart_key or 'xbar' in chart_key:
            y_data = data.values
        else:
            y_data = np.abs(np.diff(data.values, prepend=data.values[0]))

        dynamic_charts.append({
            "title": chart_title,
            "x": data.index,
            "y": y_data,
            "mean": chart_info.get('mean'),
            "upper": chart_info.get('usl', chart_info.get('ucl')),
            "lower": chart_info.get('lsl', chart_info.get('lcl'))
        })

# 4. 解析 EWMA 圖表
if len(spc_result) > 2 and isinstance(spc_result[2], dict) and 'ewma_values' in spc_result[2]:
    dynamic_charts.append({
        "title": "EWMA CONTROL CHART",
        "x": data.index,
        "y": spc_result[2]['ewma_values'],
        "mean": spc_result[0]['mean'],
        "upper": spc_result[2].get('usl', spc_result[2].get('ucl')),
        "lower": spc_result[2].get('lsl', spc_result[2].get('lcl'))
    })

# 5. 【直接帶入你提供的 CUSUM 資料】
cusum_result = {
    'c_plus': np.array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 2.18573285e+00, 9.38374203e-01, 1.10697885e+00, 5.34079144e-01, 0.00000000e+00, 5.91654275e-01, 6.74911631e-04, 1.52489930e+00, 3.29888282e-01, 1.87771881e+00, 2.12745478e+00, 1.09807957e+00, 2.25192551e+00, 1.38095971e+00, 2.89733822e+00, 2.57845609e+00, 5.40030566e-01, 1.45424522e+00, 4.29355445e-01, 1.63551140e+00, 1.58868398e+00, 1.83284166e+00, 2.55012563e+00, 2.64648361e+00, 1.68158444e+00, 1.37938048e+00, 0.00000000e+00, 2.92511302e-01, 1.34055752e-01, 1.57705661e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.69359674e+00, 1.80563866e+00, 2.46609452e+00, 1.99258589e+00, 9.57817746e-01, 1.27267996e+00, 1.17024594e+00, 1.41765046e+00, 7.06892769e-01, 3.88185652e-01, 0.00000000e+00, 9.18113491e-01, 3.01443508e-01, 5.63912479e-01, 1.94165433e+00, 2.88210869e+00, 2.69001000e+00, 2.20375124e+00, 2.47258201e+00, 2.80040218e+00, 1.34881043e+00, 1.67762288e+00, 1.59370198e+00, 9.94330684e-01, 4.84947803e-01, 0.00000000e+00, 0.00000000e+00, 1.52049739e-01, 0.00000000e+00, 4.35321070e-01, 1.39406735e+00, 1.54140883e+00, 1.62681744e+00, 1.61085324e+00, 1.10784938e+00, 4.31478580e-01, 6.08856086e-01, 0.00000000e+00, 3.37417978e-01, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 2.92102234e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.39640082e-01, 0.00000000e+00, 0.00000000e+00, 7.56260548e-01, 4.11456986e-01, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
    'c_minus': np.array([0.        , -1.49992637, -2.36208027, -2.06692322, -2.30253579, 0.        , -0.69058698,  0.        , -0.01612804, -0.30802733, 0.        , -0.03420769,  0.        , -0.63823935,  0.        , 0.        , -0.47260354,  0.        , -0.31419413,  0.        , 0.        , -1.48165386, -0.01066753, -0.47878564,  0.        , 0.        ,  0.        ,  0.        ,  0.        , -0.40812749, -0.15355979, -1.41902545, -0.56974247, -0.17142636,  0.        , -1.21982125, -1.34436956, -2.33722538, -2.36849725, -2.67748625, -3.69688981, -3.29191633, -1.04154793, -0.37273433,  0.        , 0.        , -0.47799647,  0.        ,  0.        ,  0.        , -0.15398602,  0.        ,  0.        ,  0.        , -0.05989831, 0.        ,  0.        ,  0.        ,  0.        ,  0.        , 0.        ,  0.        , -0.89482008, -0.00923595,  0.        , -0.04259963,  0.        , -0.17113278, -0.96407497, -0.25525356, 0.        ,  0.        ,  0.        ,  0.        ,  0.        , 0.        ,  0.        , -0.11959913,  0.        , -0.83344761, 0.        , -0.27860292, -0.99547016, -1.51805436, -2.95148548, -2.36550359, -1.99447617, -1.78676126, -2.11971682, -2.73904115, -3.1199355 , -4.08051825, -3.3841065 , -4.18734388, -4.13628044, -2.82324822, -2.61128011, -2.47896607, -4.15808567, -4.33477157]),
    'decision_limit': np.float64(2.7838583442708065)
}

h_limit = cusum_result['decision_limit']

# 加入 CUSUM+ 圖表
dynamic_charts.append({
    "title": "CUSUM POSITIVE CHART (C+)",
    "x": data.index,
    "y": cusum_result['c_plus'],
    "mean": 0,
    "upper": h_limit,
    "lower": 0
})

# 加入 CUSUM- 圖表（取絕對值方便跟警戒線比較觀察，或者直接用原值）
dynamic_charts.append({
    "title": "CUSUM NEGATIVE CHART (C-)",
    "x": data.index,
    "y": np.abs(cusum_result['c_minus']), # 取絕對值讓圖面在正向展示與 h 的距離
    "mean": 0,
    "upper": h_limit,
    "lower": 0
})

# 6. 動態建立畫布與格位
num_charts = len(dynamic_charts)
ncols = 2
nrows = math.ceil(num_charts / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 5 * nrows))
axes_flat = np.array([axes]).flatten() if num_charts == 1 else axes.flatten()

# 7. 透過迴圈繪製：所有圖表皆具備 Dot + Line + 顏色標記
for idx, cfg in enumerate(dynamic_charts):
    ax = axes_flat[idx]
    x_val = cfg["x"]
    y_val = cfg["y"]
    
    current_color = color[:len(x_val)] if len(color) >= len(x_val) else color
    
    # 同時畫出 Dot (scatterplot) 與 Line (lineplot)
    sns.scatterplot(x=x_val, y=y_val, ax=ax, color=current_color, s=60, zorder=5, label='Data Points')
    sns.lineplot(x=x_val, y=y_val, ax=ax, color='gray', linestyle='-', alpha=0.6, label='Trend Line')
    
    # 處理上下界與平均值
    mean_val = cfg['mean'] if isinstance(cfg['mean'], (list, np.ndarray, pd.Series)) else [cfg['mean']] * len(x_val)
    upper_val = cfg['upper'] if isinstance(cfg['upper'], (list, np.ndarray, pd.Series)) else [cfg['upper']] * len(x_val)
    lower_val = cfg['lower'] if isinstance(cfg['lower'], (list, np.ndarray, pd.Series)) else [cfg['lower']] * len(x_val)

    sns.lineplot(x=x_val, y=mean_val, ax=ax, color='green', label='Mean / CL')
    sns.lineplot(x=x_val, y=upper_val, ax=ax, color='red', linestyle='--', label='Upper Limit / Decision Limit (+h)')
    sns.lineplot(x=x_val, y=lower_val, ax=ax, color='red', linestyle='--', label='Lower Limit (0)')
    
    ax.set_title(cfg["title"])
    ax.set_xlabel("Index")
    ax.set_ylabel("Values")
    ax.legend(loc='upper right')
    ax.grid(True)

# 隱藏多餘的空白子圖
for j in range(num_charts, len(axes_flat)):
    fig.delaxes(axes_flat[j])

plt.tight_layout()
plt.show()

# MSPC

# MSPC time trajectory